# 5. Pré-processamento de Dados (*Data Preprocessing*)

Notas do curso **Machine Learning Process** — tratamento de dados antes da modelagem.

---

### Índice
1. [O que é pré-processamento?](#intro)
2. [Tipos de valores ausentes](#tipos)
3. [Detectar valores ausentes](#detectar)
4. [Técnica 1 — Remover nulos (dropna)](#drop)
5. [Técnica 2 — Média / Mediana / Moda](#media)
6. [Técnica 3 — Imputação por Regressão (IterativeImputer)](#iterative)
7. [Técnica 4 — KNN Imputer](#knn)
8. [Comparação entre técnicas](#comparacao)
9. [Resumo interativo](#resumo)

---
<a id='intro'></a>
## 1. O que é pré-processamento?

Pré-processamento é a etapa de **preparar os dados** antes de analisar ou modelar. Dados brutos raramente estão prontos para ML.

| Tarefa | O que fazer |
|---|---|
| **Valores nulos** | Identificar missing, decidir: remover, imputar ou criar flag |
| **Outliers** | IQR, z-score, regras de negócio — às vezes são o sinal importante |
| **Tipos e formatos** | Datas, categorias, IDs — garantir tipos corretos |
| **Consistência** | Mesma métrica definida igual em todas as fontes |
| **Encoding** | Transformar categorias em números para o modelo |
| **Escalonamento** | Normalizar/padronizar features para algoritmos sensíveis à escala |

> **Regra:** dados sujos no pré-processamento → EDA enganosa → modelo errado → hipóteses falsas.

---
<a id='tipos'></a>
## 2. Tipos de Valores Ausentes

Entender **por que** um valor está ausente é tão importante quanto decidir **como** tratá-lo. Existem 3 mecanismos de ausência:

---

### 2.1 MCAR — *Missing Completely At Random* (Aleatório Puro)

A ausência **não tem relação** com nenhuma variável do dataset — nem com o valor ausente em si.

**Exemplo:** um sensor de temperatura falhou aleatoriamente durante 2 horas por problema de hardware — sem padrão relacionado à temperatura real ou ao momento do dia.

**Como detectar:** comparar distribuições entre linhas com e sem o valor ausente. Se forem iguais, provavelmente é MCAR.

**Tratamento possível:** qualquer técnica funciona, inclusive `dropna` (sem viés).

---

### 2.2 MAR — *Missing At Random* (Aleatório Condicional)

A ausência **se relaciona com outras variáveis observadas**, mas **não** com o próprio valor ausente.

**Exemplo:** usuários mais jovens tendem a não preencher o campo `renda` — a ausência depende da `idade` (observável), não do valor da renda em si.

**Como detectar:** a ausência varia com outras variáveis do dataset. Análise de correlação entre `is_null` e demais features revela padrões.

**Tratamento:** imputação condicional (modelos que usam as outras variáveis, como `IterativeImputer` ou `KNNImputer`). `dropna` pode criar viés.

---

### 2.3 MNAR — *Missing Not At Random* (Não Aleatório)

A ausência **depende do próprio valor ausente** — o dado está faltando *por causa* do valor que teria.

**Exemplo:** pessoas com renda muito alta tendem a não declarar sua renda. A ausência do campo `renda` está correlacionada com *a renda real em si*.

**Como detectar:** é o mais difícil de identificar — exige conhecimento do domínio e comparação com fontes externas.

**Tratamento:** nenhuma técnica automática resolve completamente. Opções:
- Usar `add_indicator=True` para sinalizar onde foi imputado
- Coletar mais dados ou usar fontes alternativas
- Modelar separadamente os segmentos com e sem o valor

---

### Resumo dos tipos

| Tipo | A ausência depende de... | Risco de viés com dropna | Melhor tratamento |
|---|---|---|---|
| **MCAR** | Nada (puro acaso) | Baixo | Qualquer técnica |
| **MAR** | Outras variáveis observadas | Médio | Imputação condicional (KNN, MICE) |
| **MNAR** | Do próprio valor ausente | **Alto** | Indicador + coleta adicional |

---
<a id='detectar'></a>
## 3. Detectar Valores Ausentes

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Carregar dados
df = pd.read_csv('../5_data_preprocessing/5_1_missing_values/clv_data.csv', index_col=0)
df['lifetime_value'] = df['purchases'] * 20

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Contagem simples de nulos
print("Contagem de nulos por coluna:")
print(df.isnull().sum())

In [ ]:
# Tabela com percentual
def nulls_summary(df):
    resultado = pd.DataFrame({
        'null_count': df.isnull().sum(),
        'null_pct':   df.isnull().sum() / len(df),
        'dtype':      df.dtypes
    })
    return resultado.sort_values('null_pct', ascending=False)

nulls_summary(df)

In [ ]:
# Detectar padrão MAR: a ausência varia com outra variável?
df['age_missing'] = df['age'].isnull().astype(int)

print("Média de income por grupo (age presente vs. ausente):")
print(df.groupby('age_missing')[['income', 'days_on_platform']].mean())
print("\nSe as médias diferirem significativamente → indício de MAR")

df.drop(columns='age_missing', inplace=True)

---
<a id='drop'></a>
## 4. Técnica 1 — Remover Nulos (`dropna`)

Remove todas as linhas com pelo menos um `NaN`.

**Use quando:** poucos nulos (< 5%), MCAR, dataset grande o suficiente.

**Evite quando:** muitos nulos ou MAR/MNAR — cria viés na amostra.

In [ ]:
drop_df = df.copy()
print(f'Linhas antes: {len(drop_df)}')

drop_df = drop_df.dropna()
print(f'Linhas depois: {len(drop_df)}')
print(f'Removidas: {len(df) - len(drop_df)} ({(len(df)-len(drop_df))/len(df)*100:.1f}%)')

X_train_d = drop_df[['age','days_on_platform','income']][:4000]
y_train_d = drop_df['lifetime_value'][:4000]
X_test_d  = drop_df[['age','days_on_platform','income']][4000:]
y_test_d  = drop_df['lifetime_value'][4000:]

---
<a id='media'></a>
## 5. Técnica 2 — Média / Mediana / Moda

Substitui `NaN` por um resumo estatístico. **Sempre calcule no treino e aplique no teste.**

| Estatística | Quando usar |
|---|---|
| **Média** | Distribuição simétrica, sem outliers |
| **Mediana** | Distribuição assimétrica ou com outliers |
| **Moda** | Variáveis categóricas |

In [ ]:
m_df = df.copy()
X_m = m_df[['age','days_on_platform','income']]
y_m = m_df['lifetime_value']

X_train_m = X_m[:4000].copy()
y_train_m = y_m[:4000]
X_test_m  = X_m[4000:].copy()
y_test_m  = y_m[4000:]

# ── Média ──────────────────────────────────────────────────────
age_mean  = np.mean(X_train_m['age'])
days_mean = np.mean(X_train_m['days_on_platform'])

X_train_m['age']              = X_train_m['age'].fillna(age_mean)
X_test_m['age']               = X_test_m['age'].fillna(age_mean)       # usa média do TREINO
X_train_m['days_on_platform'] = X_train_m['days_on_platform'].fillna(days_mean)
X_test_m['days_on_platform']  = X_test_m['days_on_platform'].fillna(days_mean)

print(f'Média usada para age:              {age_mean:.2f}')
print(f'Média usada para days_on_platform: {days_mean:.2f}')
print(f'Nulos restantes no treino: {X_train_m.isnull().sum().sum()}')

In [ ]:
# ── Mediana ─────────────────────────────────────────────────────
age_median = np.median(df['age'].dropna())
print(f'Mediana de age: {age_median}')

# ── Moda (para categóricas) ─────────────────────────────────────
gender_mode = stats.mode(df['gender'].dropna(), keepdims=True)[0][0]
print(f'Moda de gender: {gender_mode}')

# Comparar média vs. mediana
print(f'\nMédia de age:   {df["age"].mean():.2f}')
print(f'Mediana de age: {df["age"].median():.2f}')
print('(Se diferirem muito → distribuição assimétrica → use mediana)')

---
<a id='iterative'></a>
## 6. Técnica 3 — Imputação por Regressão (`IterativeImputer` / MICE)

Usa um **modelo de ML** para prever o valor ausente com base nas outras colunas. Itera várias vezes refinando cada imputação (*Multiple Imputation by Chained Equations*).

```
Rodada 1: imputa age    → usa days_on_platform, income
          imputa income → usa age (imputado), days_on_platform
Rodada 2: reimputa age  → income está melhor → age melhora
...
Rodada N: convergência
```

| Parâmetro | O que faz |
|---|---|
| `estimator` | Modelo para prever o nulo (`BayesianRidge` default, ou `RandomForestRegressor`) |
| `max_iter` | Número de rodadas de imputação |
| `add_indicator` | Cria coluna 0/1 indicando onde foi imputado |
| `random_state` | Reprodutibilidade |

In [ ]:
r_df = df.copy()
X_r = r_df[['age','days_on_platform','income']]
y_r = r_df['lifetime_value']

X_train_r = X_r[:4000].copy()
y_train_r = y_r[:4000]
X_test_r  = X_r[4000:].copy()
y_test_r  = y_r[4000:]

# Treinar APENAS no conjunto de treino
imp = IterativeImputer(max_iter=10, random_state=0)
imp.fit(X_train_r)

X_train_r_imp = pd.DataFrame(imp.transform(X_train_r), columns=X_train_r.columns)
X_test_r_imp  = pd.DataFrame(imp.transform(X_test_r),  columns=X_test_r.columns)

print(f'Nulos no treino após IterativeImputer: {X_train_r_imp.isnull().sum().sum()}')
print(f'Nulos no teste  após IterativeImputer: {X_test_r_imp.isnull().sum().sum()}')
X_train_r_imp.describe().round(2)

---
<a id='knn'></a>
## 7. Técnica 4 — KNN Imputer

Para cada `NaN`, encontra os **K registros mais similares** (por distância Euclidiana) que têm valor naquele campo e usa a média desses vizinhos.

| Parâmetro | O que faz |
|---|---|
| `n_neighbors` | Número de vizinhos (default: 5) |
| `weights` | `"uniform"` (igual) ou `"distance"` (mais próximo = mais peso) |
| `metric` | Distância usada (default: `nan_euclidean`) |

> ⚠️ **Escale as features antes do KNN** — distância Euclidiana é sensível à escala. Uma coluna em milhares domina sobre uma em dezenas.

In [ ]:
X_train_knn = X_r[:4000].copy()
X_test_knn  = X_r[4000:].copy()

imputer_knn = KNNImputer(n_neighbors=5, weights='uniform')
imputer_knn.fit(X_train_knn)

X_train_k = pd.DataFrame(imputer_knn.transform(X_train_knn), columns=X_train_knn.columns)
X_test_k  = pd.DataFrame(imputer_knn.transform(X_test_knn),  columns=X_test_knn.columns)

y_train_k = y_r[:4000].reset_index(drop=True)
y_test_k  = y_r[4000:].reset_index(drop=True)

print(f'Nulos após KNNImputer: {X_train_k.isnull().sum().sum()}')

# Efeito de K no resultado
print('\nEfeito de K na média imputada de age:')
for k in [3, 5, 10, 20]:
    imp_k = KNNImputer(n_neighbors=k)
    imp_k.fit(X_train_knn)
    X_k = pd.DataFrame(imp_k.transform(X_train_knn), columns=X_train_knn.columns)
    print(f'  K={k:2d} → média age: {X_k["age"].mean():.2f}  desvio: {X_k["age"].std():.2f}')

---
<a id='comparacao'></a>
## 8. Comparação entre Técnicas

Treinamos um `RandomForestRegressor` com cada dataset imputado e comparamos o **MAE** (erro médio absoluto) no alvo `lifetime_value`. **Menor = melhor.**

In [ ]:
resultados = {}

# Drop Null
clf = RandomForestRegressor(n_estimators=50, random_state=0)
clf.fit(X_train_d, y_train_d)
resultados['Drop Null'] = mean_absolute_error(y_test_d, clf.predict(X_test_d))

# Média
clf2 = RandomForestRegressor(n_estimators=50, random_state=0)
clf2.fit(X_train_m, y_train_m)
resultados['Média'] = mean_absolute_error(y_test_m, clf2.predict(X_test_m))

# IterativeImputer
clf3 = RandomForestRegressor(n_estimators=50, random_state=0)
clf3.fit(X_train_r_imp, y_train_r)
resultados['Regressão Iterativa (MICE)'] = mean_absolute_error(y_test_r, clf3.predict(X_test_r_imp))

# KNN
clf4 = RandomForestRegressor(n_estimators=50, random_state=0)
clf4.fit(X_train_k, y_train_k)
resultados['KNN (k=5)'] = mean_absolute_error(y_test_k, clf4.predict(X_test_k))

# Resultado
res_df = pd.DataFrame.from_dict(resultados, orient='index', columns=['MAE'])
res_df = res_df.sort_values('MAE')
res_df['Ranking'] = range(1, len(res_df)+1)
print("Comparação de técnicas (menor MAE = melhor):")
display(res_df)

---
<a id='resumo'></a>
## 9. Resumo Interativo

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

resumos = {
    "🎲 MCAR — Aleatório Puro": [
        "Ausência não tem relação com nenhuma variável",
        "Ex: sensor que falhou aleatoriamente",
        "Baixo risco de viés com qualquer técnica",
        "dropna é seguro se o % for pequeno",
    ],
    "📊 MAR — Aleatório Condicional": [
        "Ausência depende de outras variáveis observadas",
        "Ex: jovens não preenchem renda (depende da idade)",
        "dropna cria viés — preferir imputação condicional",
        "KNNImputer e IterativeImputer lidam bem com MAR",
    ],
    "⚠️ MNAR — Não Aleatório": [
        "Ausência depende do próprio valor ausente",
        "Ex: ricos não declaram renda",
        "Nenhuma técnica resolve completamente",
        "Usar add_indicator=True + coletar mais dados",
    ],
    "🔧 Técnicas de Imputação": [
        "dropna: simples, para MCAR com poucos nulos",
        "Média/mediana/moda: rápido, comprime variância",
        "IterativeImputer (MICE): usa relações entre features, lento",
        "KNNImputer: usa vizinhos similares, sensível à escala",
    ],
    "✅ Boas Práticas": [
        "Sempre calcular estatísticas no TREINO e aplicar no teste",
        "df.copy() antes de modificar — preserve o original",
        "add_indicator=True quando padrão de ausência é informativo",
        "Comparar técnicas pelo MAE/AUC downstream, não pela teoria",
        "Escalar features antes do KNNImputer",
    ],
}

accordion = widgets.Accordion()
children = []
for titulo, pontos in resumos.items():
    itens = "\n".join([f"- {p}" for p in pontos])
    out = widgets.Output()
    with out:
        display(Markdown(itens))
    children.append(out)

accordion.children = children
for i, titulo in enumerate(resumos):
    accordion.set_title(i, titulo)

display(accordion)

In [ ]:
# Quiz rápido
print("=" * 58)
print("      QUIZ — Tipos de Missing Values e Imputação")
print("=" * 58)

quiz = [
    ("Usuários com renda alta tendem a não declarar a renda. Qual tipo?",
     "MNAR — a ausência depende do próprio valor (renda alta)"),
    ("Um sensor falhou aleatoriamente por 2h. Qual tipo?",
     "MCAR — ausência sem relação com nenhuma variável"),
    ("Usuários de iOS preenchem menos o campo 'profissão'. Qual tipo?",
     "MAR — ausência relacionada à plataforma (observável), não à profissão"),
    ("Qual técnica É arriscada para MAR e MNAR?",
     "dropna — pode criar viés ao remover linhas com padrão não aleatório"),
    ("Por que calcular a média SEMPRE no treino?",
     "Evitar data leakage — usar o teste para calcular seria ver o futuro"),
    ("Por que escalar features antes do KNNImputer?",
     "Distância Euclidiana é sensível à escala — colunas em milhares dominam o cálculo"),
]

for i, (pergunta, resposta) in enumerate(quiz, 1):
    print(f"\nQ{i}: {pergunta}")
    input("   → Pense na resposta e pressione Enter...")
    print(f"   ✅ {resposta}")

print("\n" + "=" * 58)
print("Quiz concluído!")

---
## Suas notas

- 
- 